# 🔗 Column Lineage — Test Notebook
Tests `column_lineage_tab.py` logic end-to-end inside Jupyter.  
All graph-building and HTML-rendering code is **copied verbatim** from the
generated tab file so no changes are needed to any deployed file.

---
### How to use
1. Fill in your Greenplum connection details in **Cell 2**.
2. Run all cells (`Kernel → Restart & Run All`).
3. Use the **dropdown widgets** to pick a column → table.
4. Click **Show Lineage** to render the interactive canvas.


In [ ]:
# ── Cell 2 : DB Connection Config ──────────────────────────────────────
# Edit these four values to match your environment.

GP_HOST = "greenplum-rdsp.zur.swissbank.com"
GP_PORT = 5432
GP_DB   = "gprdsp"
GP_USER = "ds_rdsp_dev"

# Schema where ikg_column_lineage_master_auto_refresh lives
IKG_SCHEMA = "sandbox_prj_smart_insights"

# ── Imports ──────────────────────────────────────────────────────────────
import json, re, os, tempfile, warnings
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, IFrame

try:
    from sqlalchemy import create_engine, text
    SQLALCHEMY_OK = True
except ImportError:
    SQLALCHEMY_OK = False
    warnings.warn("sqlalchemy not found — install it: pip install sqlalchemy psycopg2-binary")

# ── Build engine (prompts for password if not set) ────────────────────
import getpass

_pw = getpass.getpass(f"Greenplum password for {GP_USER}@{GP_HOST}:")
_engine = create_engine(
    f"postgresql+psycopg2://{GP_USER}:{_pw}@{GP_HOST}:{GP_PORT}/{GP_DB}",
    pool_pre_ping=True,
)

def _run(sql: str) -> pd.DataFrame:
    with _engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print("✅  DB connection ready")


In [ ]:
# ── Cell 3 : Data-access functions (mirror of utils/data.py) ────────────

def get_all_target_columns() -> pd.DataFrame:
    return _run(f"""
        SELECT DISTINCT target_column
        FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh
        WHERE target_column IS NOT NULL AND TRIM(target_column) <> ''
        ORDER BY target_column
    """)

def get_tables_for_column(target_column: str) -> pd.DataFrame:
    safe = str(target_column).replace("'", "''")
    return _run(f"""
        SELECT DISTINCT target_table
        FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh
        WHERE target_column = '{safe}'
          AND target_table IS NOT NULL AND TRIM(target_table) <> ''
        ORDER BY target_table
    """)

def get_column_lineage_upstream(target_column: str, target_table: str) -> pd.DataFrame:
    sc = str(target_column).replace("'", "''")
    st = str(target_table).replace("'", "''")
    return _run(f"""
        WITH RECURSIVE upstream AS (
            SELECT target_table, target_schema, target_column,
                   source_table, source_schema, source_column,
                   logic, process, path, 0 AS depth
            FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh
            WHERE target_table = '{st}' AND target_column = '{sc}'
            UNION ALL
            SELECT cl.target_table, cl.target_schema, cl.target_column,
                   cl.source_table, cl.source_schema, cl.source_column,
                   cl.logic, cl.process, cl.path, u.depth + 1
            FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh cl
            JOIN upstream u
              ON cl.target_table = u.source_table
             AND cl.target_column = u.source_column
            WHERE u.depth < 6
        )
        SELECT DISTINCT * FROM upstream
    """)

def get_column_lineage_downstream(target_column: str, target_table: str) -> pd.DataFrame:
    sc = str(target_column).replace("'", "''")
    st = str(target_table).replace("'", "''")
    return _run(f"""
        WITH RECURSIVE downstream AS (
            SELECT target_table, target_schema, target_column,
                   source_table, source_schema, source_column,
                   logic, process, path, 0 AS depth
            FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh
            WHERE source_table = '{st}' AND source_column = '{sc}'
            UNION ALL
            SELECT cl.target_table, cl.target_schema, cl.target_column,
                   cl.source_table, cl.source_schema, cl.source_column,
                   cl.logic, cl.process, cl.path, d.depth + 1
            FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh cl
            JOIN downstream d
              ON cl.source_table = d.target_table
             AND cl.source_column = d.target_column
            WHERE d.depth < 4
        )
        SELECT DISTINCT * FROM downstream
    """)

print("✅  Data functions ready")


In [ ]:
# ── Cell 4 : Graph-builder (verbatim copy from column_lineage_tab.py) ───

_NODE_W, _NODE_H = 230, 54
_COL_GAP, _ROW_GAP, _LEFT_PAD = 340, 130, 80

def _safe_str(v) -> str:
    return str(v) if v is not None and pd.notna(v) else ""

def _node_ntype(table: str, schema: str) -> str:
    t = (table  or "").lower()
    s = (schema or "").lower()
    if t.endswith("_ikg"):                          return "ikg"
    if t.startswith("temp_") or t.endswith("_tmp"): return "tmp"
    if "ikg" in s or "model" in s or "nlg" in s:   return "ikg"
    return "ext"

def build_column_lineage_payload(target_column: str, target_table: str):
    target_column = _safe_str(target_column).strip()
    target_table  = _safe_str(target_table).strip()

    df_up   = get_column_lineage_upstream(target_column, target_table)
    df_down = get_column_lineage_downstream(target_column, target_table)

    edge_tuples = []
    for df_s in [df_up, df_down]:
        if df_s.empty:
            continue
        for r in df_s.itertuples(index=False):
            src_t = _safe_str(getattr(r,"source_table",None))
            src_c = _safe_str(getattr(r,"source_column",None))
            tgt_t = _safe_str(getattr(r,"target_table",None))
            tgt_c = _safe_str(getattr(r,"target_column",None))
            logic = _safe_str(getattr(r,"logic",None))
            proc  = _safe_str(getattr(r,"process",None))
            if src_t and tgt_t:
                edge_tuples.append((src_t,src_c,tgt_t,tgt_c,logic,proc))

    if not edge_tuples:
        return ([{"id":target_table,"x":_LEFT_PAD,"y":70,"w":_NODE_W,"h":_NODE_H,
                  "role":"focal","ntype":"ikg","col_label":target_column,"level":0}],
                [], {"focal":1,"upstream":0,"downstream":0,"relations":0})

    upstream_tables, upstream_col = {}, {}
    downstream_tables, downstream_col = {}, {}

    if not df_up.empty:
        adj_up = {}
        for st,sc2,tt,tc,*_ in edge_tuples:
            adj_up.setdefault(tt,[]).append((st,sc2,tc))
        visited = {target_table}
        queue = [(target_table,0,target_column)]
        while queue:
            tbl,depth,col_in = queue.pop(0)
            for src,sc2,tc in adj_up.get(tbl,[]):
                if src not in visited:
                    visited.add(src)
                    nd = depth - 1
                    upstream_tables[src] = nd
                    upstream_col[src]    = sc2 or col_in
                    queue.append((src,nd,sc2 or col_in))

    if not df_down.empty:
        adj_down = {}
        for st,sc2,tt,tc,*_ in edge_tuples:
            adj_down.setdefault(st,[]).append((tt,sc2,tc))
        visited2 = {target_table}
        queue2 = [(target_table,0,target_column)]
        while queue2:
            tbl,depth,col_out = queue2.pop(0)
            for dst,sc2,tc in adj_down.get(tbl,[]):
                if dst not in visited2:
                    visited2.add(dst)
                    nd = depth + 1
                    downstream_tables[dst] = nd
                    downstream_col[dst]    = tc or col_out
                    queue2.append((dst,nd,tc or col_out))

    schema_map = {}
    for df_s in [df_up, df_down]:
        if df_s.empty: continue
        for cp in [("source_table","source_schema"),("target_table","target_schema")]:
            tc2,sc2 = cp
            if tc2 in df_s.columns and sc2 in df_s.columns:
                for r in df_s[[tc2,sc2]].drop_duplicates().itertuples(index=False):
                    tb,sh = _safe_str(r[0]),_safe_str(r[1])
                    if tb and tb not in schema_map: schema_map[tb]=sh

    depth_buckets = {0:[target_table]}
    for tb,d in upstream_tables.items():   depth_buckets.setdefault(d,[]).append(tb)
    for tb,d in downstream_tables.items(): depth_buckets.setdefault(d,[]).append(tb)
    min_depth = min(depth_buckets)

    nodes = []
    for depth in sorted(depth_buckets):
        ci = depth - min_depth
        for ri,tbl in enumerate(sorted(depth_buckets[depth])):
            x,y = _LEFT_PAD+ci*_COL_GAP, 60+ri*_ROW_GAP
            if depth==0:   role,col_lbl = "focal", target_column
            elif depth<0:  role,col_lbl = "upstream",   upstream_col.get(tbl,"")
            else:          role,col_lbl = "downstream", downstream_col.get(tbl,"")
            nt = role if role=="focal" else _node_ntype(tbl, schema_map.get(tbl,""))
            nodes.append({"id":tbl,"x":x,"y":y,"w":_NODE_W,"h":_NODE_H,
                           "role":role,"ntype":nt,"col_label":col_lbl,"level":ci})

    all_ids = {n["id"] for n in nodes}
    seen,edges = set(),[]
    for st,sc2,tt,tc,logic,proc in edge_tuples:
        if st not in all_ids or tt not in all_ids: continue
        k=(st,sc2,tt,tc)
        if k in seen: continue
        seen.add(k)
        edges.append({"from":st,"to":tt,"src_col":sc2,"tgt_col":tc,
                      "logic":logic if logic.lower() not in ("","none","nan") else "","process":proc})

    counts = {"focal":1,"upstream":len(upstream_tables),
              "downstream":len(downstream_tables),"relations":len(edges)}
    return nodes, edges, counts

print("✅  Graph builder ready")


In [ ]:
# ── Cell 5 : Template renderer ──────────────────────────────────────────
# Looks for the template in common locations relative to this notebook.
# Adjust TEMPLATE_PATH if your directory layout differs.

_SEARCH_PATHS = [
    Path("utils/column_lineage_template.html"),                    # notebook next to utils/
    Path("../utils/column_lineage_template.html"),                 # notebook one level down
    Path("insights_metadata_dashboard/utils/column_lineage_template.html"),
]

def _find_template(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(
        "column_lineage_template.html not found. "
        f"Searched: {[str(p) for p in paths]}. "
        "Place this notebook next to the utils/ folder or update TEMPLATE_PATH."
    )

TEMPLATE_PATH = _find_template(_SEARCH_PATHS)
print(f"✅  Template found: {TEMPLATE_PATH.resolve()}")

def render_column_lineage_html(target_column: str, target_table: str) -> str:
    template = TEMPLATE_PATH.read_text(encoding="utf-8")
    nodes, edges, counts = build_column_lineage_payload(target_column, target_table)
    payload = (
        f"const NODES  = {json.dumps(nodes)};\n"
        f"const EDGES  = {json.dumps(edges)};\n"
        f"const FOCAL_TABLE = {json.dumps(target_table)};\n"
        f"const FOCAL_COL   = {json.dumps(target_column)};\n"
        f"const CL_COUNTS   = {json.dumps(counts)};"
    )
    html_out = re.sub(
        r"const NODES\s*=\s*\[\s*\];\s*"
        r"const EDGES\s*=\s*\[\s*\];\s*"
        r"const FOCAL_TABLE\s*=\s*\".*?\";\s*"
        r"const FOCAL_COL\s*=\s*\".*?\";\s*"
        r"const CL_COUNTS\s*=\s*\{.*?\};",
        payload, template, flags=re.DOTALL,
    )
    safe = f"{target_column} in {target_table}"
    html_out = re.sub(r"<title>.*?</title>", f"<title>{safe}</title>",
                      html_out, count=1, flags=re.DOTALL)
    return html_out

print("✅  Renderer ready")


In [ ]:
# ── Cell 6 : Load columns for dropdown ──────────────────────────────────
print("Loading all target columns from DB … (may take a few seconds)")
_df_cols = get_all_target_columns()
_col_list = _df_cols["target_column"].dropna().astype(str).tolist()
print(f"✅  {len(_col_list)} columns loaded")


In [ ]:
# ── Cell 7 : Interactive widgets ────────────────────────────────────────

_style   = {"description_width": "110px"}
_layout  = widgets.Layout(width="460px")

w_col   = widgets.Dropdown(options=_col_list, description="Column:",
                            layout=_layout, style=_style)
w_table = widgets.Dropdown(options=[], description="Table:",
                            disabled=True, layout=_layout, style=_style)
w_btn   = widgets.Button(description="Show Lineage ▶",
                          button_style="primary",
                          layout=widgets.Layout(width="160px", margin="8px 0 0 0"))
w_info  = widgets.Label(value="")
w_out   = widgets.Output()

def _on_col_change(change):
    val = change["new"]
    if not val:
        w_table.options   = []
        w_table.disabled  = True
        w_info.value      = ""
        return
    df_t = get_tables_for_column(val)
    tbls = df_t["target_table"].dropna().astype(str).tolist()
    w_table.options  = tbls
    w_table.value    = tbls[0] if tbls else None
    w_table.disabled = len(tbls) == 0
    w_info.value     = f"{len(tbls)} table(s) contain '{val}'"

def _on_btn_click(b):
    w_out.clear_output()
    col   = w_col.value
    table = w_table.value
    if not col or not table:
        with w_out:
            print("⚠  Please select both a column and a table.")
        return
    with w_out:
        print(f"Building lineage for  {col}  in  {table} …")
    try:
        html_content = render_column_lineage_html(col, table)
    except Exception as exc:
        with w_out:
            print(f"❌  Error: {exc}")
        return
    # Write to a temp HTML file next to the notebook for the IFrame
    tmp = Path(tempfile.mktemp(suffix=".html", dir="."))
    tmp.write_text(html_content, encoding="utf-8")
    with w_out:
        w_out.clear_output(wait=True)
        display(IFrame(src=str(tmp), width="100%", height="700px"))

w_col.observe(_on_col_change, names="value")
w_btn.on_click(_on_btn_click)

# Trigger initial table load
if _col_list:
    _on_col_change({"new": w_col.value})

display(widgets.VBox([
    widgets.HTML("<b>Column Lineage Explorer</b>"),
    w_col,
    w_table,
    w_info,
    w_btn,
    w_out,
]))


---
### Notes
- The canvas is **fully interactive**: drag nodes, pan, scroll-zoom, click any box for the side panel.
- The iframe temp file is written to the same directory as this notebook — delete `*.html` files when done.
- To test a specific column directly (no widgets), run:
  ```python
  h = render_column_lineage_html("account_balance", "household_profile_curr_ikg")
  display(HTML(h))       # renders inline (JS canvas may need IFrame for full interactivity)
  ```
